In [ ]:

import gc
import argparse
import logging
import warnings
import torch
import traceback
import torchvision.transforms as transforms

from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN
from torchvision.datasets import ImageNet
from components.broadcast_components.broadcasting_process.broadcast_reporting_utilities import \
    BroadcastMetricGatheringUtilities
from components.broadcast_components.WZ_models.WZQuantizerWithDataPrep import QuantizerWithDataPrep
from components.broadcast_components.WZ_models.wz_quant_RNN import PL_EncoderDecoder_RNN
from components.other_utilities.user_logger import UnifiedLoggingClass


In [ ]:
import gzip
import pickle
from components.broadcast_components.broadcasting_process.CancerProt import CancerProtocol
class CustomCancerProtocol(CancerProtocol):
    def _post_reconstruction_processing(self, agent_id, worker_count, dict_shape, curr_recons_vector):
        # save curr_recons_vector to folder wi (i=agent_id), and name it round_j (j=current round)
        # then call super()._post_reconstruction_processing to do the rest
        # compress and save
        path_to_file = r'D:\User\App Files\Projects\VUB-ACS-25_Thesis\experiments\run_sim_script\past recons list\\'+\
                       f'w{agent_id+1}\\round_{self.curr_round_id}.pkl.gz'
        with gzip.open(path_to_file, 'wb', compresslevel=1) as f:
            pickle.dump((curr_recons_vector, dict_shape), f)
        return super()._post_reconstruction_processing(agent_id, worker_count, dict_shape, curr_recons_vector)


In [ ]:
data_folder = r'../../data'

dataset = [
    FasterSVHN(

        # limit_count = 10000,

        root=data_folder + '/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]
num_classes = 10
worker_count=5
batch_size = 5000
wz_model = PL_EncoderDecoder_RNN(inp_dim=1, side_info_size=0, num_planes=2, bins_per_plane=16, tau=1.3,
                                 reconst_ld=400, lr=1e-3, tau_rate=10, marginal=True).to(torch.float32)
wz_model.load_state_dict(torch.load(f'{data_folder}/basicRNN_2plane_4bins_state.pt', map_location='cpu'))

base_quantizer = QuantizerWithDataPrep(wz_model, train_sample_size=300_000,
                                       count_side_info_data=0, enable_progress_bar=False)

broadcast_prot_base = CustomCancerProtocol(worker_count, base_quantizer,)
broadcast_prot_base.no_global_quantization = True
model = ResNetPLModel(num_classes=num_classes, resnet_version='resnet18', lr=0.001, )
model.load_state_dict(torch.load(f'{data_folder}/resnet18_svhn.pth', map_location='cpu'))
sim = FLSimulator(
    pl_model=model, num_agents=worker_count, communication_rounds=80, client_epochs_per_round=20,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False)
sim.run_simulation(broadcast_prot_base)